In [3]:
import random
def mark_aleatorizado(secuencia, k):
  cache = list(range(k))
  marcado = [False] * k
  fallos = 0
  hits = 0
  fase = 1
  log = []

  for t, pagina in enumerate(secuencia):
    if pagina in cache:
      idx = cache.index(pagina)
      marcado[idx] = True
      hits += 1
      log.append((t, pagina, 'HIT', None, None, fase))
    else:
      fallos += 1
      no_marcados = [i for i, m in enumerate(marcado) if not m]

      if not no_marcados:
        fase += 1
        marcado = [False] * k
        no_marcados = list(range(k))
        log.append((t, pagina, 'NUEVA_FASE', None, None, fase))

      slot = random.choice(no_marcados)
      expulsada = cache[slot]
      cache[slot] = pagina
      marcado[slot] = True
      log.append((t, pagina, 'MISS', expulsada, slot, fase))

  return fallos, hits, log, fase


def comparar_politicas(secuecia, k, n_trials=500):
  def lru(seq,k):
    cache, orden, f = [], [], 0
    for p in seq:
      if p in cache:
        orden.remove(p); orden.append(p)
      else:
        f += 1
        if len(cache) == k:
          v = orden.pop(0); cache.remove(v)
        cache.append(p); orden.append(p)
    return f

  def fifo(seq, k):
    from collections import deque
    cache, q, f = set(), deque(), 0
    for p in seq:
      if p not in cache:
        f += 1
        if len(cache) == k:
          old = q.popleft(); cache.remove(old)
        cache.add(p); q.append(p)

    return f


  mark_resultados  = [mark_aleatorizado(secuecia, k)[0] for _ in range(n_trials)]
  lru_f = lru(secuecia, k)
  fifo_f = fifo(secuecia, k)
  mark_avg = sum(mark_resultados) / n_trials


  print(f"Caché k={k}, |secuencia|={len(secuecia)}")
  print(f"  LRU  (determinista): {lru_f} fallos")
  print(f"  FIFO (determinista): {fifo_f} fallos")
  print(f"  MARK (aleatorizado): {mark_avg:.1f} fallos (promedio {n_trials} runs)")
  print(f"  Ratio MARK/LRU:      {mark_avg/lru_f:.2f}x  (cota teórica: {sum(1/i for i in range(1,k+1)):.2f}x)")


# --- Demo ---
if __name__ == "__main__":
    random.seed(42)
    N, k, T = 12, 4, 40
    secuencia = [random.randint(0, N-1) for _ in range(T)]

    fallos, hits, log, fases = mark_aleatorizado(secuencia, k)
    print(f"MARK: {fallos} fallos, {hits} aciertos, {fases} fases\n")

    for t, pag, evento, exp, slot, fase in log[:15]:
        if evento == 'HIT':
            print(f"  t={t:2d}  p{pag}  HIT")
        elif evento == 'MISS':
            print(f"  t={t:2d}  p{pag}  MISS → expulsada p{exp} (slot {slot})")
        else:
            print(f"  t={t:2d}  p{pag}  ── NUEVA FASE {fase} ──")

    print()
    comparar_politicas(secuencia, k)


MARK: 26 fallos, 14 aciertos, 9 fases

  t= 0  p10  MISS → expulsada p2 (slot 2)
  t= 1  p1  HIT
  t= 2  p0  HIT
  t= 3  p11  MISS → expulsada p3 (slot 3)
  t= 4  p4  ── NUEVA FASE 2 ──
  t= 4  p4  MISS → expulsada p1 (slot 1)
  t= 5  p3  MISS → expulsada p10 (slot 2)
  t= 6  p3  HIT
  t= 7  p2  MISS → expulsada p0 (slot 0)
  t= 8  p11  HIT
  t= 9  p1  ── NUEVA FASE 3 ──
  t= 9  p1  MISS → expulsada p2 (slot 0)
  t=10  p10  MISS → expulsada p3 (slot 2)
  t=11  p11  HIT
  t=12  p8  MISS → expulsada p4 (slot 1)

Caché k=4, |secuencia|=40
  LRU  (determinista): 29 fallos
  FIFO (determinista): 32 fallos
  MARK (aleatorizado): 29.2 fallos (promedio 500 runs)
  Ratio MARK/LRU:      1.01x  (cota teórica: 2.08x)
